In [1]:
import pandas as pd
df= pd.read_csv("../data/processed/clean_student_lifestyle.csv")
print(df.head())

   Age  Gender   Department  CGPA  Sleep_Duration  Study_Hours  \
0   22  Female      Science  3.50             7.3          3.3   
1   20    Male  Engineering  2.72             5.5          7.2   
2   20    Male      Medical  3.01             5.4          2.3   
3   21    Male  Engineering  3.63             8.1          2.0   
4   19    Male         Arts  3.14             6.8          2.6   

   Social_Media_Hours  Physical_Activity  burnout  
0                 3.4                114        0  
1                 6.0                142        0  
2                 1.8                137        0  
3                 4.6                130        0  
4                 4.3                  4        1  


In [2]:
X= df.drop(columns=["burnout"])
y= df["burnout"]

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [4]:
X_train.dtypes

Age                     int64
Gender                    str
Department                str
CGPA                  float64
Sleep_Duration        float64
Study_Hours           float64
Social_Media_Hours    float64
Physical_Activity       int64
dtype: object

In [5]:
numeric_features = [
    "Age",
    "CGPA",
    "Sleep_Duration",
    "Study_Hours",
    "Social_Media_Hours",
    "Physical_Activity"
]

categorical_features = [
    "Gender",
    "Department"
]

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [7]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline= Pipeline(steps= [("preprocessor",preprocessor),("model",LogisticRegression(max_iter=1000))])

In [8]:
logistic_pipeline.fit(X_train, y_train)
y_pred_logistic = logistic_pipeline.predict(X_test)


In [9]:
from sklearn.metrics import classification_report, roc_auc_score

print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred_logistic))
print("ROC AUC Score for Logistic Regression:", roc_auc_score(y_test, logistic_pipeline.predict_proba(X_test)[:, 1]))

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.87      0.98      0.92     16822
           1       0.61      0.20      0.30      3178

    accuracy                           0.85     20000
   macro avg       0.74      0.59      0.61     20000
weighted avg       0.83      0.85      0.82     20000

ROC AUC Score for Logistic Regression: 0.8123781385804004


In [10]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        eval_metric="logloss",
        random_state=42
    )
}

In [11]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

results={}

for model_name, model in models.items():
    pipeline= Pipeline(
        steps=[
            ("preprocessor",preprocessor),
            ("model",model)
        ]
    )
    pipeline.fit(X_train, y_train)
    preds= pipeline.predict(X_test)
    probs= pipeline.predict_proba(X_test)[:, 1]
    roc_auc= roc_auc_score(y_test, probs)

    print(f"Model Name: {model_name}")
    print("Classification Report:")
    print(classification_report(y_test, preds))
    print(f"ROC AUC Score: {roc_auc}")
    print("-" * 50)
    results[model_name]= roc_auc


Model Name: Random Forest
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.93     16822
           1       0.73      0.39      0.51      3178

    accuracy                           0.88     20000
   macro avg       0.81      0.68      0.72     20000
weighted avg       0.87      0.88      0.86     20000

ROC AUC Score: 0.885750955905311
--------------------------------------------------
Model Name: XGBoost
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.93     16822
           1       0.73      0.39      0.51      3178

    accuracy                           0.88     20000
   macro avg       0.81      0.68      0.72     20000
weighted avg       0.87      0.88      0.86     20000

ROC AUC Score: 0.8835100600602511
--------------------------------------------------


In [12]:
print(results)

{'Random Forest': 0.885750955905311, 'XGBoost': 0.8835100600602511}


In [13]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

In [14]:
rf_pipeline= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])
rf_param_grid = {
    "model__n_estimators": [100, 200, 300, 400, 600],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}
rf_random_search= RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=20,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    verbose=1,
    n_jobs=-1
)
rf_random_search.fit(X_train, y_train)
print("Best Hyperparameters for Random Forest:", rf_random_search.best_params_)
best_rf_model= rf_random_search.best_estimator_
rf_preds= best_rf_model.predict(X_test)
rf_probs= best_rf_model.predict_proba(X_test)[:, 1]
print("Classification Report for Random Forest:", classification_report(y_test, rf_preds))
print("ROC AUC Score for Random Forest:", roc_auc_score(y_test, rf_probs))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Hyperparameters for Random Forest: {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_depth': 30}
Classification Report for Random Forest:               precision    recall  f1-score   support

           0       0.89      0.97      0.93     16822
           1       0.74      0.40      0.51      3178

    accuracy                           0.88     20000
   macro avg       0.82      0.68      0.72     20000
weighted avg       0.87      0.88      0.87     20000

ROC AUC Score for Random Forest: 0.8839533608443317


In [15]:
xg_pipeline= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(random_state=42, eval_metric="logloss"))
])
xg_param_grid={
    "model__n_estimators": [100, 200, 300, 400, 600],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 6, 9],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}
xg_random_search= RandomizedSearchCV(
    estimator=xg_pipeline,
    param_distributions=xg_param_grid,
    n_iter=20,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    verbose=1,
    n_jobs=-1
)
xg_random_search.fit(X_train, y_train)
print("Best Hyperparameters for XGBoost:", xg_random_search.best_params_)
best_xg_model= xg_random_search.best_estimator_
xg_preds= best_xg_model.predict(X_test)
xg_probs= best_xg_model.predict_proba(X_test)[:, 1]
print("Classification Report for XGBoost:", classification_report(y_test, xg_preds))
print("ROC AUC Score for XGBoost:", roc_auc_score(y_test, xg_probs))

Fitting 3 folds for each of 20 candidates, totalling 60 fits


/Users/rachanadutta/projects/burnout-risk-prediction/venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Hyperparameters for XGBoost: {'model__subsample': 0.8, 'model__n_estimators': 600, 'model__max_depth': 9, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
Classification Report for XGBoost:               precision    recall  f1-score   support

           0       0.89      0.97      0.93     16822
           1       0.73      0.39      0.51      3178

    accuracy                           0.88     20000
   macro avg       0.81      0.68      0.72     20000
weighted avg       0.87      0.88      0.86     20000

ROC AUC Score for XGBoost: 0.8843746920613039


In [16]:
rf_model = rf_random_search.best_estimator_.named_steps["model"]
feature_names = rf_random_search.best_estimator_.named_steps[
    "preprocessor"
].get_feature_names_out()


In [17]:
import pandas as pd

importance = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
})

feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
)

feature_importance_df.head(10)

,feature,importance
2,num__Sleep_Duration,0.412616
5,num__Physical_Activity,0.251047
1,num__CGPA,0.097451
3,num__Study_Hours,0.091935
4,num__Social_Media_Hours,0.084287
0,num__Age,0.032526
11,cat__Department_Medical,0.004742
8,cat__Department_Arts,0.004702
12,cat__Department_Science,0.004569
10,cat__Department_Engineering,0.004568
